# Step 1: viscosity fix + sloshing-frequency validation (Colab T4)

Runs two GPU-bound but *cheap* experiments from `SPH_Enhancement_Guide_18Sept2026.zip` (see `the development log` Entry 87) — both at this project's EXISTING particle resolution (`dx=0.01`, no resolution increase), so each takes minutes, not hours, on a free T4:

1. **Experiment A** — replay a real recorded trajectory (`tank_motion_trained_seed0.csv`) through `SPHBackend` twice: once at the current `viscosity=1.0` (a real, longstanding bug — 1000x real water's dynamic viscosity), once at the corrected `viscosity=0.001`. Compares the fluid response directly.
2. **Experiment B** — a NEW trajectory (`tank_motion_free_oscillation.csv`, generated by `scripts/generate_free_oscillation_trajectory.py`, already committed) that gives the tank a brief, smooth kick then holds it perfectly still, so the fluid's OWN free sloshing frequency can be measured via FFT and compared against the analytical prediction (this is genuinely different from the existing `tank_motion_decaying_sine.csv`, which continuously forces the tank and would just show its own 0.4 Hz driving frequency back, not a real measurement).

**Before running:** `Runtime -> Change runtime type -> Hardware accelerator -> GPU`, then Save.

In [ ]:
!nvidia-smi

## Load the Chrono::FSI build

A working build should already exist on Google Drive (`MyDrive/chrono-fsi-build`, per `the project notes`'s Phase 13-14 section) — this avoids a multi-hour from-source rebuild. If the check below says `False`, stop here and follow `documentation/colab_fsi_build.md` to build from source first (a separate, longer task), then come back.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
CHRONO_ROOT = '/content/drive/MyDrive/chrono-fsi-build'  # adjust if your build lives elsewhere
install_path = f'{CHRONO_ROOT}/chrono-install'
found = os.path.isdir(install_path)
print('Existing Chrono::FSI build found at', install_path, ':', found)
if not found:
    print()
    print('No existing build found here. See documentation/colab_fsi_build.md to build')
    print('from source first -- do not continue past this cell until that path exists.')

In [ ]:
# These do NOT persist on Drive even though the build itself does (the project notes, Entry 37) --
# expect to re-run this in every fresh session.
!apt-get install -y -qq libirrlicht1.8 libpython3.12

## Preload + import (the exact sequence confirmed working, the project notes Phase 13-14)

If any `ctypes.CDLL` call below fails, read the real error and preload ONLY the library it names — do not loop over every `.so` in the lib folder (a real, documented way to crash the whole Colab kernel, per `the project notes`).

In [ ]:
import ctypes
ctypes.CDLL(f"{CHRONO_ROOT}/chrono-install/lib/libyaml-cpp.so.0.8", mode=ctypes.RTLD_GLOBAL)
ctypes.CDLL(f"{CHRONO_ROOT}/chrono-install/lib/libChrono_core.so", mode=ctypes.RTLD_GLOBAL)
ctypes.CDLL(f"{CHRONO_ROOT}/chrono-install/lib/libChrono_fsi.so", mode=ctypes.RTLD_GLOBAL)
ctypes.CDLL(f"{CHRONO_ROOT}/chrono-install/lib/libChrono_fsisph.so", mode=ctypes.RTLD_GLOBAL)
ctypes.CDLL(f"{CHRONO_ROOT}/chrono-install/lib/libChrono_irrlicht.so", mode=ctypes.RTLD_GLOBAL)
import sys
sys.path.insert(0, f"{CHRONO_ROOT}/chrono-install/share/chrono/python")
import pychrono, pychrono.fsi, pychrono.irrlicht
print("pychrono.fsi import OK:", pychrono.fsi)

## Get this project's code

If the repo is private and `git clone` fails, comment that cell out and use the upload cell instead with `LiquidSloshingRL_Full_Project.zip` (already on your machine).

In [ ]:
!git clone https://github.com/RacerRed505/LiquidSloshingRL.git
%cd LiquidSloshingRL

In [ ]:
# Fallback if git clone isn't accessible from this Colab session -- upload the zip instead:
# from google.colab import files
# uploaded = files.upload()  # pick LiquidSloshingRL_Full_Project.zip
# !unzip -q -o LiquidSloshingRL_Full_Project.zip -d /content/LiquidSloshingRL
# %cd /content/LiquidSloshingRL

In [ ]:
# Verified against the REAL import chain (physics/sph_backend.py,
# builders/fluid_builder.py, config/simulation_config.py, utilities/*) --
# this notebook's two experiments only need numpy + pychrono + stdlib.
# gymnasium/stable-baselines3/scipy are NOT imported anywhere in that path
# and are deliberately NOT installed here (avoids pulling in torch's own
# CUDA runtime via stable-baselines3 for no reason, alongside pychrono's).
# numpy/matplotlib are already in the base Colab image.
print('No extra pip installs needed for this notebook -- numpy/pychrono only.')

## Experiment A — the `viscosity=1.0` bug vs. the physical `viscosity=0.001` fix

Same trajectory, same resolution (`dx=0.01`), only `--viscosity` differs. Both runs use
`scripts/run_sph_backend_trajectory.py`'s new `--viscosity` flag (added alongside this
notebook, the development log Entry 87) — the class default stays `1.0` everywhere else in the
repo, so nothing outside this notebook is affected.

**Real risk, stated plainly, not glossed over:** dropping viscosity by 1000x removes a lot
of numerical damping. This project has a real history of SPH solver crashes
(`particle derivVel is NAN`) from parameter changes that reduced damping/stability margin
(the development log Entries 46, 58). **If the `--viscosity 0.001` cell below crashes or the
kernel dies, that is itself a real, useful result** — it would mean this tank's SPH setup
is more numerically sensitive to physical viscosity than the guide assumed, worth writing
up honestly in `the development log` rather than treated as a notebook bug. If it happens, try a
smaller change first (e.g. `--viscosity 0.1`, then `0.01`) to find where it actually breaks,
rather than jumping straight back to `1.0`.

In [ ]:
!python -m scripts.run_sph_backend_trajectory tank_motion_trained_seed0.csv \
    --output sph_output_trained_seed0_viscosity_baseline.csv

In [ ]:
!python -m scripts.run_sph_backend_trajectory tank_motion_trained_seed0.csv \
    --output sph_output_trained_seed0_viscosity_fixed.csv \
    --viscosity 0.001

In [ ]:
import csv
import numpy as np

def load_com_x(path):
    rows = list(csv.DictReader(open(path)))
    return np.array([float(r['sph_fluid_com_x']) for r in rows])

baseline = load_com_x('sph_output_trained_seed0_viscosity_baseline.csv')
fixed = load_com_x('sph_output_trained_seed0_viscosity_fixed.csv')
n = min(len(baseline), len(fixed))
diff = baseline[:n] - fixed[:n]
print(f'Steps compared: {n}')
print(f'Mean |difference| in fluid_com_x: {np.mean(np.abs(diff)):.6f} m')
print(f'Max  |difference| in fluid_com_x: {np.max(np.abs(diff)):.6f} m')
print(f'Baseline (viscosity=1.0)   final com_x: {baseline[-1]:.6f} m')
print(f'Fixed    (viscosity=0.001) final com_x: {fixed[-1]:.6f} m')

## Experiment B — sloshing-frequency validation

`tank_motion_free_oscillation.csv` is already committed to the repo (generated and sanity-checked locally against `PendulumBackend`'s own analytical frequency before this notebook was written — its FFT recovered 1.30 Hz against a known 1.3429 Hz, ~3% off, within one FFT frequency bin for a 10s/500-step signal). This cell regenerates it only if missing.

In [ ]:
!test -f tank_motion_free_oscillation.csv || python -m scripts.generate_free_oscillation_trajectory --output tank_motion_free_oscillation.csv

In [ ]:
!python -m scripts.run_sph_backend_trajectory tank_motion_free_oscillation.csv \
    --output sph_output_free_oscillation.csv \
    --viscosity 0.001

In [ ]:
import csv
import numpy as np

rows = list(csv.DictReader(open('sph_output_free_oscillation.csv')))
com_x = np.array([float(r['sph_fluid_com_x']) for r in rows])

control_dt = 0.02  # must match generate_free_oscillation_trajectory's own control_dt
t = np.arange(len(com_x)) * control_dt

# FIXED (found on the RTX 3090 machine, the development log Entry 95): a naive full-series FFT
# here is WRONG, not just imprecise -- this trajectory's real fluid damps out within ~4s and
# sits flat for the remaining ~6s of the 10s window, so a full-series FFT is dominated by that
# long flat tail and reports a spurious ~0.1 Hz "peak" (92.7% error, confirmed on a real run).
# Fixed by measuring the period directly from peak-to-peak spacing in the genuinely-oscillating
# early window instead -- far more reliable with only ~7 real cycles than an FFT's frequency
# resolution allows anyway (also confirmed on the real run: a 4.0s-windowed FFT still gave a
# noisy, wrong answer due to non-integer-cycle spectral leakage at this short a window).
peaks = [
    i for i in range(2, min(250, len(com_x) - 1))
    if com_x[i] > com_x[i - 1] and com_x[i] > com_x[i + 1] and com_x[i] > com_x.mean()
]
assert len(peaks) >= 2, 'need at least 2 peaks to measure a period -- inspect com_x directly if this fires'
periods = np.diff(t[peaks])
mean_period_s = periods.mean()
f_peak_hz = 1.0 / mean_period_s
omega_peak = 2 * np.pi * f_peak_hz
print(f'Peaks found at t={np.round(t[peaks], 3)} s')
print(f'Periods between consecutive peaks: {np.round(periods, 4)} s (mean {mean_period_s:.4f} s)')

# Analytical shallow-water potential-flow prediction (the guide's own Part 4 Phase D formula),
# using this project's real interior tank dimensions -- a DIFFERENT formula from
# PendulumBackend's own Housner-equivalent-pendulum frequency (both are legitimate, not the
# same model -- shown separately below rather than conflated).
L, h0, g = 0.388, 0.20, 9.81
k1 = np.pi / L
omega_analytical = np.sqrt(g * k1 * np.tanh(k1 * h0))
error_pct = abs(omega_peak - omega_analytical) / omega_analytical * 100

print(f'SPH-measured natural frequency (peak-spacing):  {f_peak_hz:.4f} Hz ({omega_peak:.4f} rad/s)')
print(f'Analytical, shallow-water potential flow:        -       ({omega_analytical:.4f} rad/s)')
print(f'PendulumBackend/Housner-pendulum (reference):    1.3429 Hz (8.4376 rad/s)')
print(f'SPH vs. shallow-water analytical error:          {error_pct:.1f}%  (guide target: <5%)')

## Download results back to your machine

In [ ]:
from google.colab import files
import shutil, os

outputs = [
    'sph_output_trained_seed0_viscosity_baseline.csv',
    'sph_output_trained_seed0_viscosity_fixed.csv',
    'sph_output_free_oscillation.csv',
    'tank_motion_free_oscillation.csv',
]
os.makedirs('/content/step1_results', exist_ok=True)
for f in outputs:
    shutil.copy(f, f'/content/step1_results/{f}')
shutil.make_archive('/content/step1_results', 'zip', '/content/step1_results')
files.download('/content/step1_results.zip')

## Next steps

1. Extract `step1_results.zip` into the project root on your local machine.
2. Add the real numbers from Experiments A/B to `the development log` Entry 87 (or a new entry), same as every other real finding in this project.
3. If the viscosity fix or the frequency result look meaningfully different from what's expected, that's a real finding worth its own entry, not something to quietly adjust away.
4. Commit the new CSVs if you want them as permanent project data (matches the convention of `tank_motion_decaying_sine.csv`/`sph_output_trained_seed0.csv` already being committed).